# Pairwise numerical statistics
Counts for one-to-many or selected directed pairs. Optional target-repeat matching preserves the pairwise boundary rule; it does not normalize reversed boundaries. No figures or phylogenetic interpretation. See `../README.md` or `../README.ru.md`. Run cells in order only after selecting real result paths.

In [ ]:
from pathlib import Path
import json
import sys
import csv
from collections import Counter
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "bin" / "annotation_io.py").is_file():
    raise RuntimeError("Start Jupyter from the repository or its notebooks directory")
sys.path.insert(0, str(ROOT / "bin"))
from annotation_io import iter_bed

In [ ]:
GENOMES_TSV = ROOT / "examples/genomes.tsv"
PAIRS_TSV = ROOT / "examples/pairs.tsv"
RESULTS_DIR = ROOT / "results"
OUTPUT_DIR = RESULTS_DIR / "analysis"
SEARCH_GAPS = [50, 55, 65, 75, 85, 100]
VALID_TRANSFER_STATUSES = {
    "occupied_expected_length", "occupied_different_length",
    "local_indel_or_rearranged", "empty_with_TSD", "empty_without_TSD",
}
CHUNKSIZE = 250_000

MATCH_REPEATS = False
INCLUDE_HAL = False

In [ ]:
def manifest_path(manifest, value):
    path = Path(value)
    return path if path.is_absolute() else manifest.parent / path

GENOMES = pd.read_csv(GENOMES_TSV, sep="\t", dtype=str, keep_default_na=False).set_index("genome_id", verify_integrity=True)
if not GENOMES.index.to_series().str.fullmatch(r"[A-Za-z][A-Za-z0-9_]*").all():
    raise ValueError("Invalid genome_id")
PAIRS = pd.read_csv(PAIRS_TSV, sep="\t", dtype=str, keep_default_na=False)
if PAIRS.duplicated(["source", "target"]).any():
    raise ValueError("Duplicate directed pair")
if not set(PAIRS.source).union(PAIRS.target).issubset(GENOMES.index):
    raise ValueError("Pair manifest refers to an unknown genome")

def statuses_path(pair):
    supplied = getattr(pair, "statuses", "")
    return manifest_path(PAIRS_TSV, supplied) if supplied else RESULTS_DIR / "pairs" / f"{pair.source}__to__{pair.target}" / "statuses.tsv"

def bed_input(genome_id):
    row = GENOMES.loc[genome_id]
    published = RESULTS_DIR / "genomes" / genome_id / "repeats.bed"
    if published.is_file():
        return published, "bed6", {}
    if row.get("annotation_format", "") in {"bed6", "custom5"} and row.get("annotation", ""):
        chrom_map = {}
        if row.get("chrom_map", ""):
            chrom_map = json.loads(manifest_path(GENOMES_TSV, row.chrom_map).read_text())
        return manifest_path(GENOMES_TSV, row.annotation), row.annotation_format, chrom_map
    raise FileNotFoundError(f"No BED annotation for {genome_id}; export repeats.bed first")

def read_repeats(genome_id):
    path, format_name, chrom_map = bed_input(genome_id)
    return pd.DataFrame(
        [(chrom_map.get(x.chrom, x.chrom), x.start0, x.end0, x.strand, x.rep_name)
         for x in iter_bed(path, format_name)],
        columns=["chr", "start", "end", "strand", "rep_name"],
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def match_pairwise_repeats(row, rep_by_chr):
    # Preserve the pairwise scenario: do not normalize the two inner boundaries.
    if row.status not in VALID_TRANSFER_STATUSES:
        return -1, None, None
    repeats = rep_by_chr.get(str(row.target_contig))
    if repeats is None:
        return 0, [], []
    starts, ends, indices, names = repeats
    for gap in SEARCH_GAPS:
        if row.target_strand == "+":
            left, right = row.left_target_end - gap, row.right_target_start + gap
        elif row.target_strand == "-":
            left, right = row.right_target_end - gap, row.left_target_start + gap
        else:
            raise ValueError(f"Unexpected strand: {row.target_strand}")
        mask = (starts >= left) & (ends <= right)
        matched_indices, matched_names = indices[mask].tolist(), names[mask].tolist()
        if matched_indices:
            return len(matched_indices), matched_indices, matched_names
    return 0, [], []

In [ ]:
status_rows, confidence_rows, hal_rows = [], [], []
for pair in tqdm(list(PAIRS.itertuples(index=False)), desc="Pairs"):
    counts, confidence = Counter(), Counter()
    seen = set()
    rep_by_chr = None
    if MATCH_REPEATS:
        repeats = read_repeats(pair.target)
        rep_by_chr = {
            chrom: (g.start.to_numpy(), g.end.to_numpy(), g.index.to_numpy(), g.rep_name.to_numpy())
            for chrom, g in repeats.groupby("chr", sort=False)
        }
    pair_dir = OUTPUT_DIR / f"{pair.source}__to__{pair.target}"
    pair_dir.mkdir(exist_ok=True)
    match_output = pair_dir / "annotation_matches.tsv"
    first_chunk = True
    for chunk in pd.read_csv(statuses_path(pair), sep="\t", chunksize=CHUNKSIZE,
                             dtype={"group_id": str, "repeat_ids_inside": str}, keep_default_na=False):
        ids = chunk.group_id.tolist()
        if len(set(ids)) != len(ids) or seen.intersection(ids):
            raise ValueError("Duplicate group_id; refusing to drop scientific records")
        seen.update(ids)
        counts.update(chunk.status)
        confidence.update(chunk.confidence)
        if MATCH_REPEATS:
            for col in ["left_target_start", "left_target_end", "right_target_start", "right_target_end"]:
                chunk[col] = pd.to_numeric(chunk[col].replace("", np.nan), errors="raise")
            matches = [match_pairwise_repeats(row, rep_by_chr) for row in chunk.itertuples(index=False)]
            table = chunk[["group_id", "status", "repeat_ids_inside"]].copy()
            table["n_matched_repeats"] = [x[0] for x in matches]
            table["matched_indices"] = [json.dumps(x[1]) for x in matches]
            table["matched_names"] = [json.dumps(x[2]) for x in matches]
            table.to_csv(match_output, sep="\t", index=False, mode="w" if first_chunk else "a", header=first_chunk)
            first_chunk = False
    status_rows.extend({"source":pair.source,"target":pair.target,"status":k,"count":v} for k,v in counts.items())
    confidence_rows.extend({"source":pair.source,"target":pair.target,"confidence":k,"count":v} for k,v in confidence.items())
    if INCLUDE_HAL:
        hal_path = RESULTS_DIR / "validation" / f"{pair.source}__to__{pair.target}" / "hal_statuses.tsv"
        hal = pd.read_csv(hal_path, sep="\t", usecols=["group_id", "hal_status"], dtype=str)
        if hal.group_id.duplicated().any():
            raise ValueError("Duplicate HAL group_id")
        joined = pd.DataFrame({"group_id": sorted(seen)}).merge(hal, on="group_id", how="left", validate="one_to_one")
        if not set(hal.group_id).issubset(seen):
            raise ValueError("HAL table contains loci absent from the selected status table")
        joined["hal_status"] = joined.hal_status.fillna("not_validated")
        joined.to_csv(pair_dir / "hal_by_group.tsv", sep="\t", index=False)
        hal_rows.extend({"source":pair.source,"target":pair.target,"hal_status":k,"count":v}
                        for k,v in joined.hal_status.value_counts().items())
status_counts = pd.DataFrame(status_rows, columns=["source","target","status","count"])
status_counts.to_csv(OUTPUT_DIR / "status_counts.tsv", sep="\t", index=False)
pd.DataFrame(confidence_rows).to_csv(OUTPUT_DIR / "confidence_counts.tsv", sep="\t", index=False)
if INCLUDE_HAL:
    pd.DataFrame(hal_rows).to_csv(OUTPUT_DIR / "hal_counts.tsv", sep="\t", index=False)
status_counts